# Moving Averages and Trends

## What you'll learn
- Smoothing noisy prices with a rolling average
- Plotting price against its moving averages
- A simple trend signal from two moving averages

In [ ]:
import pandas as pd

df = pd.read_csv("data/stock_history.csv", parse_dates=["Date"])
df = df.drop_duplicates().sort_values("Date")
# One price column per ticker, indexed by date:
closes = df.pivot_table(index="Date", columns="Ticker", values="Close")
closes = closes.ffill()   # carry the last known price over any gaps
print(closes.head())

## Rolling (moving) averages

Daily prices are noisy. A **moving average** smooths them by averaging the last N days,
revealing the underlying trend. `rolling(window).mean()` does exactly this. Short windows
react quickly; long windows are smoother.

In [ ]:
atlas = closes["ATLAS"]
sma_20 = atlas.rolling(20).mean()
sma_50 = atlas.rolling(50).mean()
print(sma_20.tail())

## Plotting price with its averages

Seeing the price against its 20-day and 50-day averages is the classic trend view.

In [ ]:
import matplotlib.pyplot as plt

ax = atlas.plot(label="ATLAS close", figsize=(9, 4), alpha=0.5)
sma_20.plot(ax=ax, label="20-day average")
sma_50.plot(ax=ax, label="50-day average")
ax.set_title("ATLAS price with moving averages")
ax.set_ylabel("Price")
ax.legend()
plt.tight_layout()
plt.show()

## A simple trend signal

A common rule of thumb: when the short average rises above the long average, the trend is
up ("bullish"); when it falls below, the trend is down. We can mark those states with a
boolean and count how often each occurs.

In [ ]:
uptrend = sma_20 > sma_50
print("Days in an uptrend:", int(uptrend.sum()))
print("Days in a downtrend:", int((~uptrend & sma_50.notna()).sum()))

Detecting the exact days the signal **flips** (a "crossover") is just where the boolean
changes from the day before.

In [ ]:
crossovers = uptrend.ne(uptrend.shift())
flip_dates = uptrend.index[crossovers.fillna(False)]
print("Number of crossovers:", len(flip_dates))
print(flip_dates[:5].tolist())

## Practice

1. Build clean wide `closes` and pick the `HELIOS` series.
2. Compute its 10-day and 30-day moving averages.
3. Plot the price with both averages, titled and labelled.
4. Count how many days the 10-day average is above the 30-day average.
5. Compute a 50-day moving average of `Volume` for one ticker (reload the long data for
   volume).

In [ ]:
# Your practice code here

## Summary

- `series.rolling(window).mean()` is a moving average that smooths noise to show a trend.
- Shorter windows react faster; longer windows are smoother.
- Comparing a short and a long average gives a simple up/down trend signal.
- A crossover is where that comparison flips from the previous day.

## Practice Solutions

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("data/stock_history.csv", parse_dates=["Date"]).drop_duplicates().sort_values("Date")
closes = df.pivot_table(index="Date", columns="Ticker", values="Close").ffill()

# 1, 2.
helios = closes["HELIOS"]
sma_10 = helios.rolling(10).mean()
sma_30 = helios.rolling(30).mean()

# 3.
ax = helios.plot(label="HELIOS", figsize=(9, 4), alpha=0.5)
sma_10.plot(ax=ax, label="10-day")
sma_30.plot(ax=ax, label="30-day")
ax.set_title("HELIOS with moving averages")
ax.legend()
plt.tight_layout()
plt.show()

# 4.
print("Days 10d above 30d:", int((sma_10 > sma_30).sum()))

# 5.
vol = df[df["Ticker"] == "ATLAS"].set_index("Date")["Volume"]
print(vol.rolling(50).mean().tail())